**Manual pipeline run - step 0 of 7** (`0_Data_Quality_Check.ipynb`)

Run All. When it finishes, open **`1_Long_Files.ipynb`**. Nothing has to be typed in - every notebook in
this folder runs top to bottom on its own.

A copy of `codes\data quality\Compendium_Data_Quality.ipynb`, made by
`make_manual_pipeline_run.py`. Change that notebook, not this copy,
then re-run the script.

# Data quality — the pipeline's first step

Part 1 runs before notebook 1, on the raw questionnaires and on
`external data\<Chapter>\`, and stops the pipeline there.

Reads every raw questionnaire, the same files notebook 1 reads, and finds
what is worth a human's judgement before a full run:

* **labels the dictionary has never seen** — a column header or a value with
  no good match at any confidence, in either language;
* **Value cells that are not a number and that `clean_values()` could not
  make sense of on its own** — everything it *can* resolve (a unit phrase, a
  placeholder, a sum) happens automatically in the real run and needs no
  review; this is only the residue;
* **structural problems** — reporting only: a duplicate column header, a
  stray space in a merge key, a sheet with no `index` row, breaks parsing
  itself and has to be fixed in the source Excel file by hand;
* **external data's own structural problems** - a sheet notebook 3 would
  refuse, read the same way notebook 3 reads it, on
  `external data\<Chapter>\*.xlsx`.

Nothing is changed here. Running this notebook writes one file per chapter,
**`Data quality issues before pipeline execution_<Chapter>.txt`** — every
outstanding issue, read-only:

1. Run this notebook. It writes one file per chapter, beside the codes
   folder.
2. Open the chapter(s) you are about to run and read what it found.
3. Correct whatever it calls for directly — the source questionnaire,
   `translation dict_V2.xlsx` (in the V2 root), or `value corrections.xlsx` (in
   `DATA COLLECTOR\`) — there is no intermediate file to edit and apply.
4. Tell Claude to resume. Notebook 1 runs next.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config

In [ ]:
"""
CELL: Configuration - the same paths and constants notebook 1 uses, since this
notebook has to see exactly what notebook 1 would see.
"""
DATA_COLLECTOR_PATH = Path("C:/RAFFI/DSS-local") / "DATA COLLECTOR"
# Read the same way notebook 3 reads it, for the external-data structural
# check below - a folder per chapter, not every chapter has one.
EXTERNAL_DATA_PATH = DATA_COLLECTOR_PATH / "external data"
VALUE_CORRECTIONS_PATH = DATA_COLLECTOR_PATH / "value corrections.xlsx"
COMPENDIUM_PATH = Path("C:/RAFFI/DSS-local") / "COMPENDIUM ARAB SOCIETY - V2"
TRANSLATION_DICT_PATH = COMPENDIUM_PATH / "translation dict_V2.xlsx"
# Source citations already translated - kept out of the dictionary, read
# here so a citation notebook 1 would translate is not flagged.
SOURCE_TRANSLATIONS_PATH = COMPENDIUM_PATH / "source translations_V2.xlsx"

# Part 1 writes one report per chapter here: Logs\Population_Logs\.
LOGS_ROOT = COMPENDIUM_PATH / "Logs"


def chapter_logs_folder(chapter):
    """Population -> Logs/Population_Logs."""
    return LOGS_ROOT / f"{chapter}_Logs"

# Every questionnaire, Arabic or English, one subfolder per chapter - the same
# folder notebook 1 reads. A file's language is read off its own headers.
QUESTIONNAIRES_PATH = DATA_COLLECTOR_PATH / "datacollector_received_quest"
DEFAULT_LANGUAGE = "AR"

# Leave as None to check every chapter found, or restrict e.g. ["Population"].
CHAPTERS = None

# A fuzzy match must score at least this well (0 to 1) to be used automatically
# by the real pipeline. Below it is exactly what this notebook reviews - same
# number as notebook 1's, because a label just above the line needs no review
# and one just below it does.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]

# Never fuzzy-matched by the real pipeline either: two citations differing by
# one digit score high enough to overwrite each other. A Source value still
# gets reviewed here if the dictionary has no *exact* translation for it, but
# never with a fuzzy-matched suggestion pre-filled.
COLUMNS_NOT_FUZZY_MATCHED = ["المصدر"]



## The dictionary

In [ ]:
"""
CELL: Load translation dict_V2.xlsx - identical to notebook 1's own loader, since
a label reviewed here has to be checked against the exact same vocabulary
notebook 1 will use.
"""


def load_dictionary():
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what a raw label is checked against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name_for(english_name, language):
    """Any dictionary column, spelled for the given language."""
    if language == "EN":
        return english_name
    english_to_arabic = {en: ar for ar, en in DICTIONARY_AR_TO_EN[0].items()}
    return english_to_arabic.get(english_name, english_name)


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


def known_english_columns():
    """Every English column name the dictionary knows - the vocabulary a
    column-name finding's correction has to be one of."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return sorted(set(arabic_to_english.values()))


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


def load_source_translations():
    """Every citation already translated, keyed by (chapter, exact Arabic
    text) - the same file and keys notebook 1 reads, so a citation it would
    translate is not flagged here."""
    if not SOURCE_TRANSLATIONS_PATH.exists():
        return set()
    table = pd.read_excel(SOURCE_TRANSLATIONS_PATH, engine="openpyxl")
    return {(None if pd.isna(c) else str(c).strip(), str(v).strip())
            for c, v, e in zip(table["chapter"], table["val_ar"], table["val_en"])
            if pd.notna(v) and pd.notna(e)}


SOURCE_TRANSLATED = load_source_translations()


def source_on_file(chapter, text):
    """True if source translations_V2.xlsx already has this citation."""
    text = str(text).strip()
    return (chapter, text) in SOURCE_TRANSLATED or (None, text) in SOURCE_TRANSLATED

## Recognizing a Value cell notebook 1 could not make sense of

In [ ]:
"""
CELL: clean_one_value() - identical to notebook 1's, so "not a number" means
the same thing here as it will on the real run.

Only clean_one_value() itself is needed, not the table-level clean_values():
this notebook checks one raw cell at a time as it reads each sheet, rather
than a whole reshaped column at once.
"""

PLACEHOLDERS = {"-", "--", "---", "..", "...", "n/a", "na", "n.a.", "nil", "none"}

NUMBER_IN_TEXT = re.compile(r"[-+]?\d[\d\s, ]*(?:\.\d+)?")

SUM_EXPRESSION = re.compile(r"^\d[\d\s,]*(?:\.\d+)?(?:\s*\+\s*\d[\d\s,]*(?:\.\d+)?)+$")

UNIT_MULTIPLIERS = [
    (re.compile(r"بالمليون|بالملايين"), 1_000_000, "in millions"),
    (re.compile(r"بالال?ف"), 1_000, "in thousands"),
]


def load_value_corrections():
    """Raw Value text a person has already confirmed a reading for, from a
    previous run of this same notebook's review-and-apply loop. A cell that
    matches one is not a new finding - it is already answered, and
    clean_one_value() below returns that answer instead of flagging it again.
    """
    if not VALUE_CORRECTIONS_PATH.exists():
        return {}
    table = pd.read_excel(VALUE_CORRECTIONS_PATH, engine="openpyxl")
    corrections = {}
    for chapter, raw, corrected in zip(
        table["chapter"], table["raw_value"], table["corrected_value"]
    ):
        if pd.isna(chapter) or pd.isna(raw) or pd.isna(corrected):
            continue
        corrections[(str(chapter).strip(), str(raw).strip())] = str(corrected).strip()
    return corrections


VALUE_CORRECTIONS = load_value_corrections()
if VALUE_CORRECTIONS:
    logger.info(f"Value corrections already on file: {len(VALUE_CORRECTIONS):,} "
                f"(those cells are already answered and will not be reviewed again)")


def unit_multiplier(text):
    for pattern, factor, meaning in UNIT_MULTIPLIERS:
        if pattern.search(text):
            return factor, meaning
    return 1, None


def scaled_text(number, factor):
    value = float(number) * factor
    return str(int(value)) if value.is_integer() else str(round(value, 10))


def clean_one_value(raw, chapter=None):
    """Return (cleaned, note, factor) - see notebook 1's cell of the same name
    for what each means. This copy exists so this notebook needs nothing from
    notebook 1 at import time; keep the two in step by hand if either changes.
    """
    if pd.isna(raw):
        return raw, None, 1

    text = str(raw).replace("\xa0", " ").strip()
    if text == "":
        if str(raw) == "":
            return raw, None, 1
        return "", f"{str(raw)!r} holds only whitespace - means no data, blanked", 1

    plain = re.sub(r"[\s,]", "", text)
    try:
        float(plain)
        return raw, None, 1
    except ValueError:
        pass

    confirmed = VALUE_CORRECTIONS.get((chapter, text))
    if confirmed is not None:
        return confirmed, f"{text!r} -> {confirmed} (confirmed reading, from value corrections.xlsx)", 1

    if text.lower() in PLACEHOLDERS:
        return "", f"{text!r} means no data - blanked", 1

    if SUM_EXPRESSION.match(text):
        parts = [float(re.sub(r"[\s,]", "", part)) for part in text.split("+")]
        total = scaled_text(sum(parts), 1)
        return total, (f"{text!r} -> {total} (read as a sum: "
                       f"{' + '.join(f'{part:g}' for part in parts)})"), 1

    match = NUMBER_IN_TEXT.search(text)
    if match:
        number = re.sub(r"[\s, ]", "", match.group(0))
        try:
            float(number)
        except ValueError:
            return raw, f"{text!r} is not a number and was left as it is", 1
        removed = text.replace(match.group(0), "").strip()

        factor, meaning = unit_multiplier(removed) if removed else (1, None)
        if factor != 1:
            value = scaled_text(number, factor)
            return value, (f"{text!r} -> {value} (dropped {removed!r}, meaning {meaning} "
                           f"- multiplied by {factor:,})"), factor

        return number, (f"{text!r} -> {number} (dropped {removed!r})"
                        if removed else f"{text!r} -> {number}"), 1

    return raw, f"{text!r} is not a number and was left as it is", 1


def needs_review(raw, chapter=None):
    """True only for the residue clean_one_value() could not make sense of -
    everything else is either already fine or already handled automatically
    by the real pipeline, and reviewing it here would just be noise.
    """
    cleaned, note, _ = clean_one_value(raw, chapter)
    return note is not None and str(cleaned) == str(raw)


## The three checks

In [ ]:
"""
CELL: check_labels(), check_values(), check_structure() - the three checks,
run once per sheet as it is read. Nothing here corrects anything; each only
records what it finds.
"""

# Raw hits, one per unique value per sheet - the same granularity
# correct_with_dictionary() itself works at. Aggregated across the whole run
# into one finding per (chapter, column, raw text) before the review file is
# written, so a label repeated across fifty sheets appears once.
LABEL_HITS = []
VALUE_HITS = []
STRUCTURE_PROBLEMS = []


def best_match(text, choices):
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def check_labels(table, language, chapter, file_name, sheet_name):
    """Column names and values with no good match in the dictionary - the
    same two passes as notebook 1's correct_with_dictionary(), except nothing
    is renamed and only the below-cutoff cases are kept. A label notebook 1
    would fix on its own needs no review; this is only what it would give up
    on and leave as it found it.
    """
    known_columns, known_values_by_column = vocabulary(language)
    never_guessed = [column_name(c, language) for c in COLUMNS_NOT_FUZZY_MATCHED]

    for column in table.columns:
        if column in known_columns:
            continue

        match, score = best_match(column, known_columns.keys())
        if match is None or score < FUZZY_MATCH_CUTOFF:
            LABEL_HITS.append({
                "scope": "column", "chapter": chapter, "col_ar": None,
                "raw": column, "language": language,
                "closest": match, "closest_en": None, "score": score,
                "file": file_name, "sheet": sheet_name,
            })

    for column in table.columns:
        if column in never_guessed:
            # Source is only ever a translation concern on the ARABIC side -
            # an English questionnaire's citation is already the target
            # language and needs nothing. Even there, plenty of Arabic
            # questionnaires cite an English source ("MICS 2022", a URL) -
            # already correct as it stands, exactly like notebook 4's own
            # find_untranslated() treats it. What is left after both of those
            # is flagged, but never with a fuzzy-matched suggestion: two
            # citations differing by one digit would otherwise score high
            # enough to suggest one for the other.
            if language == "AR":
                known_values = known_values_by_column.get(column) or {}
                for value in table[column].dropna().unique():
                    if (value in known_values or not looks_arabic(value)
                            or source_on_file(chapter, value)):
                        continue
                    LABEL_HITS.append({
                        "scope": "source", "chapter": chapter, "col_ar": column,
                        "raw": value, "language": language,
                        "closest": None, "closest_en": None, "score": None,
                        "file": file_name, "sheet": sheet_name,
                    })
            continue

        known_values = known_values_by_column.get(column)
        if not known_values:
            continue  # not a dictionary column, or has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue

            match, score = best_match(value, known_values.keys())
            if match is None or score < FUZZY_MATCH_CUTOFF:
                LABEL_HITS.append({
                    "scope": "value", "chapter": chapter, "col_ar": column,
                    "raw": value, "language": language,
                    "closest": match, "closest_en": known_values.get(match) if match else None,
                    "score": score, "file": file_name, "sheet": sheet_name,
                })


def check_values(table, language, chapter, file_name, sheet_name):
    """Value cells that are not a number and that clean_one_value() could not
    make sense of on its own - see needs_review() above for exactly which
    cases that is.
    """
    value_column = column_name(VALUE_COLUMN, language)
    if value_column not in table.columns:
        return

    indicator_column = column_name_for("Indicator", language)
    country_column = column_name_for("Country", language)
    year_column = column_name(YEAR_COLUMN, language)

    for position, raw in table[value_column].items():
        if pd.isna(raw) or not needs_review(raw, chapter):
            continue
        where = {}
        for field, column in [("country", country_column),
                              ("indicator", indicator_column),
                              ("year", year_column)]:
            if column in table.columns:
                where[field] = table.at[position, column]
        VALUE_HITS.append({
            "chapter": chapter, "raw": str(raw).replace("\xa0", " ").strip(),
            "file": file_name, "sheet": sheet_name, **where,
        })


# Cover / metadata tabs have no "index" header row and are skipped by design
# - not a problem, just not a data sheet. Matched by name because there is
# nothing else to go on: a tab with no header row looks structurally
# identical whether it is an expected cover page or a genuinely broken data
# sheet, and only the name tells the two apart (a data sheet in an older
# layout has no "index" row either, and IS a real problem). The metadata tab's name varies by file
# (a leading "_", a trailing " SDG", "_sdg" vs "_SDG", "المفصلة" vs
# "المجمعة") but always contains this core phrase, so it is matched as a
# substring rather than a fixed set of exact names.
COVER_TAB_NAMES = {"العنوان", "قائمة الجداول", "كيفية الإستخدام"}
COVER_TAB_MARKER = "البيانات الوصفية"


def is_cover_tab(sheet_name):
    name = str(sheet_name).strip()
    return name in COVER_TAB_NAMES or COVER_TAB_MARKER in name


def check_structure_error(chapter, language, file_name, sheet_name, error):
    """A sheet (or file) that could not be read at all."""
    STRUCTURE_PROBLEMS.append({
        "chapter": chapter, "language": language, "file": file_name,
        "sheet": sheet_name, "problem": "cannot read the sheet",
        "detail": f"{type(error).__name__}: {error}",
    })


def check_structure_raw(raw_sheet, header_rows, chapter, language, file_name, sheet_name):
    """Problems that do not raise an exception but corrupt the sheet anyway.
    A duplicate column header does not crash
    extract_tables(); it silently produces a stray extra column instead, which
    is exactly why this cannot wait to be caught as an error. Reporting only:
    every one of these has to be fixed in the source Excel file, not here.
    """
    merge_keys = {"السنة", "المؤشر", "الدولة", "Year", "Indicator", "Country"}
    here = {"chapter": chapter, "language": language, "file": file_name, "sheet": sheet_name}

    names = raw_sheet.iloc[header_rows[0]].dropna().astype(str).str.strip().tolist()

    duplicated = {n for n in names if names.count(n) > 1}
    if duplicated:
        STRUCTURE_PROBLEMS.append({**here, "problem": "duplicate column header",
                                   "detail": f"{sorted(duplicated)} - the second becomes a stray column"})

    year_like = [n for n in names if re.fullmatch(r"\d{4}(\.\d+)?", n) and not n.isdigit()]
    if year_like:
        STRUCTURE_PROBLEMS.append({**here, "problem": "year column is not a plain number",
                                   "detail": f"{year_like} - will be treated as a label, not unpivoted"})

    if not any(n.isdigit() for n in names):
        STRUCTURE_PROBLEMS.append({**here, "problem": "no year columns", "detail": "nothing to unpivot"})

    positions = raw_sheet.iloc[header_rows[0]]
    for position, name in positions.items():
        if not isinstance(name, str) or name.strip() not in merge_keys:
            continue
        if name != name.strip():
            STRUCTURE_PROBLEMS.append({**here, "problem": "whitespace in a merge-key header",
                                       "detail": f"{name!r} - breaks the source merge"})
        body = raw_sheet.index[raw_sheet[0] == "1"]
        values = raw_sheet.loc[body, position].dropna()
        untidy = [v for v in values.unique() if isinstance(v, str) and v != v.strip()]
        if untidy:
            STRUCTURE_PROBLEMS.append({**here, "problem": "whitespace in merge-key values",
                                       "detail": f"{len(untidy)} value(s), e.g. {untidy[0]!r}"})


## External data structural check

Notebook 3 reads `external data\<Chapter>\*.xlsx` - published tables that
never went through a questionnaire. Every sheet carries two blocks, each under
its own `index` header row: `index = 1` the data, `index = 2` its citation. A
sheet in any other shape is refused whole by notebook 3's
`read_external_sheet()` and reported - see notebook 3's own intro cell.

This cell runs that exact same reading here, before notebook 1 ever runs, so a
person sees what notebook 3 would refuse up front rather than discovering it
mid-pipeline. Read-only, same as everything else in Part 1 - nothing is
extracted to a long file and no reshaped copy is written from here; that stays
notebook 3's job.

In [ ]:
"""
CELL: check_external_structure() - notebook 3's own sheet reading, run here
as a read-only structural check, before the rest of the pipeline.

Duplicated from Compendium_3_External_Data.ipynb rather than imported - the
same reason save_inconsistencies() is duplicated across every pipeline
notebook: each notebook is meant to be opened and run on its own, with
nothing to import between them. Keep this in sync with notebook 3's own
infer and extract cells if its reading ever changes.
"""

SEX_LABELS = {"male": "Male", "female": "Female", "total": "Both sexes"}


def is_blank(value):
    # pd.read_excel gives a blank cell as float NaN, not None - is_blank()
    # used to miss that case, so a genuinely empty Source cell came back
    # as the literal text "nan" (str(float("nan"))) instead of staying blank.
    return pd.isna(value) or (isinstance(value, str) and value.strip() == "")


def clean_text(value):
    return "" if is_blank(value) else str(value).strip()


def collapse_whitespace(text):
    """A citation or indicator name wrapped across lines in its source
    cell, read back as one cell with an embedded newline - flattened to one
    line so it never silently breaks a downstream regex (`.` does not cross
    a newline without re.DOTALL) or reads oddly in a chart title."""
    return re.sub(r"\s+", " ", str(text)).strip()


def find_index_header_rows(raw_sheet):
    """Both rows whose own first cell reads exactly "index" - the data
    block's header, then the source block's header. A sheet with any other
    count is not this shape at all and is refused by the caller."""
    return raw_sheet.index[
        raw_sheet[0].apply(lambda v: clean_text(v).lower()) == "index"
    ].tolist()


def split_blocks(raw_sheet, header_rows):
    """Each header row's own values, and the body rows between it and the
    next header (or the end of the sheet) - the block's index numbers are
    read from that body by the caller, not assumed from position."""
    blocks = []
    for i, header_row in enumerate(header_rows):
        header = [clean_text(h) for h in raw_sheet.iloc[header_row].tolist()]
        end = header_rows[i + 1] if i + 1 < len(header_rows) else len(raw_sheet)
        body = raw_sheet.iloc[header_row + 1: end]
        blocks.append((header, body))
    return blocks


def read_data_block(header, body):
    """The index=1 block, in whichever of the two known shapes it is:

      - index, Country, Year, <the indicator's own name as this column's
        header>
      - index, Indicator, Country, Year, Male, Female, Total
      - index, Indicator, Country, Year, <a single, inconsistently named
        value column - "value", "Value", ...>

    Returns (rows, note). An unrecognized shape returns no rows and a note
    explaining why - refusing the whole sheet, since there is no longer a
    partial column to leave out on its own.
    """
    body = body[pd.to_numeric(body[0], errors="coerce") == 1]
    if body.empty:
        return [], "no index=1 rows found"

    if header[1].lower() == "country":
        indicator = collapse_whitespace(header[3])
        rows = []
        for _, row in body.iterrows():
            year = pd.to_numeric(row[2], errors="coerce")
            value = pd.to_numeric(row[3], errors="coerce")
            if pd.isna(year) or pd.isna(value):
                continue
            rows.append({"Country": clean_text(row[1]), "Year": int(year),
                        "Value": float(value), "Indicator": indicator, "Sex": None})
        return rows, f"{len(rows):,} row(s), indicator named by its own column"

    if header[1].lower() == "indicator":
        rest = header[4:]
        rest_lower = [c.lower() for c in rest]
        if sorted(rest_lower) == ["female", "male", "total"]:
            positions = {c.lower(): 4 + i for i, c in enumerate(rest)}
            rows = []
            for _, row in body.iterrows():
                year = pd.to_numeric(row[3], errors="coerce")
                if pd.isna(year):
                    continue
                indicator = collapse_whitespace(row[1])
                for label, position in positions.items():
                    value = pd.to_numeric(row[position], errors="coerce")
                    if pd.isna(value):
                        continue
                    rows.append({"Country": clean_text(row[2]), "Year": int(year),
                                "Value": float(value), "Indicator": indicator,
                                "Sex": SEX_LABELS[label]})
            return rows, f"{len(rows):,} row(s), Male/Female/Total"
        if len(rest) == 1:
            rows = []
            for _, row in body.iterrows():
                year = pd.to_numeric(row[3], errors="coerce")
                value = pd.to_numeric(row[4], errors="coerce")
                if pd.isna(year) or pd.isna(value):
                    continue
                rows.append({"Country": clean_text(row[2]), "Year": int(year),
                            "Value": float(value),
                            "Indicator": collapse_whitespace(row[1]), "Sex": None})
            return rows, f"{len(rows):,} row(s), single value column {rest[0]!r}"
        return [], (f"index=1 header has an unrecognized shape after "
                    f"Indicator/Country/Year: {rest!r} - skipped")

    return [], f"index=1 header does not start with Country or Indicator: {header!r} - skipped"


def read_source_block(header, body):
    """The index=2 block - always index, indicator, year, source. No
    Country of its own, so one citation applies to every country reporting
    that indicator-year - matched to the data block on (Indicator, Year)."""
    body = body[pd.to_numeric(body[0], errors="coerce") == 2]
    if body.empty:
        return pd.DataFrame(columns=["Indicator", "Year", "Source"])
    return pd.DataFrame({
        "Indicator": [collapse_whitespace(v) for v in body[1]],
        "Year": pd.to_numeric(body[2], errors="coerce"),
        "Source": [None if is_blank(v) else clean_text(v) for v in body[3]],
    }).dropna(subset=["Year"])


def read_external_sheet(raw_sheet, sheet_name, file_name):
    """Returns (rows, note). `rows` is a list of dicts ready to become a
    DataFrame, empty if this sheet was refused entirely; `note` explains
    what happened, for the run's report.

    Refusing the whole sheet is the only outcome for a shape this does not
    recognize - there is no header-shape guess left to make, so there is
    nothing partial to salvage the way an inferred shape sometimes had.
    """
    header_rows = find_index_header_rows(raw_sheet)
    if len(header_rows) != 2:
        return [], (f'expected 2 "index" header rows (one per block), '
                    f"found {len(header_rows)} - skipped")

    blocks = split_blocks(raw_sheet, header_rows)
    data_rows, data_note = read_data_block(*blocks[0])
    if not data_rows:
        return [], data_note

    source_table = read_source_block(*blocks[1])
    data_table = pd.DataFrame(data_rows)
    data_table["Year"] = data_table["Year"].astype(float)
    merged = data_table.merge(source_table, on=["Indicator", "Year"], how="left")
    merged["Year"] = merged["Year"].astype(int)

    rows = merged.to_dict("records")
    with_source = sum(1 for r in rows if pd.notna(r["Source"]))
    note = f"{data_note} - {with_source:,} row(s) matched to a source citation"
    return rows, note


def external_chapters():
    """Chapters this run should check for external data - those with a
    folder under external data\\ holding at least one .xlsx, restricted the
    same way chapters_to_process() restricts the questionnaire check."""
    if not EXTERNAL_DATA_PATH.exists():
        return []
    found = []
    for chapter in chapters_to_process():
        folder = EXTERNAL_DATA_PATH / chapter
        if folder.exists() and any(
            f for f in folder.glob("*.xlsx")
            if not f.name.startswith("~$") and not f.stem.endswith("_reshaped")
        ):
            found.append(chapter)
    return found


def check_external_structure(chapter):
    """Every sheet notebook 3 would refuse entirely, added to
    STRUCTURE_PROBLEMS the same way a questionnaire structural problem is -
    so it shows up in this chapter's own Data quality issues before pipeline
    execution_<Chapter>.txt with no changes needed to that writer.
    """
    folder = EXTERNAL_DATA_PATH / chapter
    files = sorted(
        f for f in folder.glob("*.xlsx")
        if not f.name.startswith("~$") and not f.stem.endswith("_reshaped")
    )
    for file_path in files:
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            STRUCTURE_PROBLEMS.append({
                "chapter": chapter, "language": "EXTERNAL", "file": file_path.name, "sheet": "-",
                "problem": "external file could not be opened",
                "detail": f"{type(error).__name__}: {error}",
            })
            continue

        for sheet_name in xls.sheet_names:
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None)
            except Exception as error:
                STRUCTURE_PROBLEMS.append({
                    "chapter": chapter, "language": "EXTERNAL", "file": file_path.name,
                    "sheet": sheet_name, "problem": "external sheet could not be read",
                    "detail": f"{type(error).__name__}: {error}",
                })
                continue

            rows, note = read_external_sheet(raw_sheet, sheet_name, file_path.name)
            if not rows:
                STRUCTURE_PROBLEMS.append({
                    "chapter": chapter, "language": "EXTERNAL", "file": file_path.name,
                    "sheet": sheet_name, "problem": "external sheet would be refused",
                    "detail": note,
                })

## Reading a raw sheet

In [ ]:
"""
CELL: Reading a raw sheet - identical to notebook 1's extract/detect/reshape
cells, so this notebook checks exactly the table notebook 1 would build,
before any correction is applied to it. No file is written from here.
"""


def discover_chapters():
    """Every chapter folder in datacollector_received_quest holding at least
    one .xlsx - identical to notebook 1's own function of the same name."""
    if not QUESTIONNAIRES_PATH.exists():
        logger.warning(f"{QUESTIONNAIRES_PATH} does not exist - nothing to read")
        return []
    return sorted(
        child.name for child in QUESTIONNAIRES_PATH.iterdir()
        if child.is_dir() and any(
            f for f in child.glob("*.xlsx") if not f.name.startswith("~$")
        )
    )


def chapters_to_process():
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters discovered on disk: {found}")
    return found


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it. Identical to
    notebook 1's own function of the same name."""
    return any("؀" <= character <= "ۿ" for character in str(text))


def detect_language(*tables):
    """Arabic as soon as one column name holds an Arabic letter, English only
    when none does - notebook 1's rule. Give it every data table in a file to
    decide the whole file."""
    names = [str(c) for table in tables for c in table.columns if not str(c).isdigit()]
    if not names:
        return DEFAULT_LANGUAGE
    return "AR" if any(looks_arabic(name) for name in names) else "EN"


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    data_columns = raw_sheet.iloc[data_header_row].dropna().str.strip()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna().str.strip()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


def reshape_and_merge(data_table, source_table, language):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name=column_name(YEAR_COLUMN, language),
        value_name=column_name(VALUE_COLUMN, language),
    )

    merge_columns = [column_name(c, language) for c in MERGE_COLUMNS]
    merge_columns = [c for c in merge_columns if c in long_table.columns and c in source_table.columns]

    long_table = long_table.copy()
    source_table = source_table.copy()
    for column in merge_columns:
        long_table[column] = long_table[column].astype(str).str.replace("\xa0", " ").str.strip()
        source_table[column] = source_table[column].astype(str).str.replace("\xa0", " ").str.strip()

    return pd.merge(long_table, source_table, on=merge_columns, how="left")


def header_table(raw_sheet, header_rows):
    """The data table's column names, as an empty table for detect_language() -
    the same names notebook 1's extract_tables() keeps, "index" left out."""
    names = raw_sheet.iloc[header_rows[0]].dropna().astype(str).str.strip()
    return pd.DataFrame(columns=[name for name in names if name != "index"])


def raw_sheets(chapter):
    """Every (file_name, sheet_name, language, table) notebook 1 would build
    for one chapter folder, in the order it would build them - each file in
    the language its own headers say, decided the same way notebook 1 does.

    A sheet that cannot be read at all, or that reads but is structurally
    broken (duplicate header, stray whitespace, ...), is reported through
    check_structure_error() / check_structure_raw() and skipped - nothing is
    yielded for it, so callers only ever see a usable table.
    """
    folder = QUESTIONNAIRES_PATH / chapter
    if not folder.exists():
        return

    for file_path in sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$")):
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            check_structure_error(chapter, "-", file_path.name, "-", error)
            continue

        # Every sheet first, so the whole file's headers decide its language
        # before any one sheet is read in it.
        sheets = []
        for sheet_name in xls.sheet_names:
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)
                header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
            except Exception as error:
                check_structure_error(chapter, "-", file_path.name, sheet_name, error)
                continue

            if len(header_rows) < 2:
                if not is_cover_tab(sheet_name):
                    # No "index" row and not a recognized cover-tab name: this
                    # is what notebook 1 itself would hit as an IndexError at
                    # its "extract" step - a real problem, not an expected skip.
                    STRUCTURE_PROBLEMS.append({
                        "chapter": chapter, "language": "-",
                        "file": file_path.name, "sheet": sheet_name,
                        "problem": "no index header row, and not a recognized cover tab",
                        "detail": f"only {len(header_rows)} 'index' row(s) found "
                                  f"(need 2) - needs a look in the source file",
                    })
                continue
            sheets.append((sheet_name, raw_sheet, header_rows))

        if not sheets:
            continue
        language = detect_language(*(header_table(raw, rows) for _, raw, rows in sheets))

        for sheet_name, raw_sheet, header_rows in sheets:
            check_structure_raw(raw_sheet, header_rows, chapter, language,
                                file_path.name, sheet_name)
            try:
                data_table, source_table = extract_tables(raw_sheet)
                table = reshape_and_merge(data_table, source_table, language)
            except Exception as error:
                check_structure_error(chapter, language, file_path.name, sheet_name, error)
                continue

            yield file_path.name, sheet_name, language, table


## run_checks()

In [ ]:
"""
CELL: run_checks() - read every questionnaire and collect every finding.
"""


def run_checks():
    LABEL_HITS.clear()
    VALUE_HITS.clear()
    STRUCTURE_PROBLEMS.clear()

    chapters = chapters_to_process()
    print(f"Reading questionnaires from: {QUESTIONNAIRES_PATH}")

    sheets_read = 0
    for step_number, chapter in enumerate(chapters, start=1):
        bar = "#" * step_number + "-" * (len(chapters) - step_number)
        print(f"[{bar}] {step_number}/{len(chapters)}  {chapter}")
        for file_name, sheet_name, language, table in raw_sheets(chapter):
            check_labels(table, language, chapter, file_name, sheet_name)
            check_values(table, language, chapter, file_name, sheet_name)
            sheets_read += 1

    logger.info(f"Read {sheets_read:,} data sheet(s)")

    ext_chapters = external_chapters()
    if ext_chapters:
        print(f"\n=== external data  ({ext_chapters}) ===")
        for chapter in ext_chapters:
            check_external_structure(chapter)

    return sheets_read


## write_chapter_reports() - one brief, read-only file per chapter


In [ ]:
"""
CELL: format_locations() and write_chapter_reports() - Part 1's one output, a
short read-only file per chapter.

Nothing here is ever applied automatically, and nothing here is aggregated
across chapters (the same mistyped column header in both Housing and Poverty
gets its own line in each chapter's own file, not one merged entry - a
per-chapter file that silently dropped the other chapter's copy would be
misleading). It exists to answer one question per chapter: what got flagged,
on which row, and why did it need a person instead of the real pipeline's own
matching? A person reads it, corrects whatever it calls for directly (the
source questionnaire, translation dict_V2.xlsx, or value corrections.xlsx), and
tells Claude to resume.

Label and structural findings are about a column, a source citation, or a
whole sheet - not one data row - so Country/Indicator/Year are printed as "-"
for those; only a value finding has a row to point at.
"""


def format_locations(locations, limit=3):
    """A short, readable sample of where a finding was seen.

    Each location is (file, sheet) for a value finding, or (chapter, file,
    sheet) for a label finding - only the last two elements are shown, since
    a label's chapter is already named on its own line.
    """
    shown = list(dict.fromkeys(locations))  # de-duplicate, keep order
    text = ", ".join(f"{loc[-2]} ({loc[-1]})" for loc in shown[:limit])
    if len(shown) > limit:
        text += f", and {len(shown) - limit} more"
    return text


CHAPTER_REPORT_PREFIX = "Data quality issues before pipeline execution_"


def issue_reason(finding):
    """Why this finding needed a person, not the real pipeline's own matching."""
    if "problem" in finding:
        return f"{finding['problem']} - {finding['detail']}"
    if finding.get("scope") == "source":
        return ("no translation in source translations_V2.xlsx, and Source is "
                "never fuzzy-matched - two "
                "citations differing by one digit would otherwise be treated as "
                "the same")
    if "scope" in finding:
        if finding["closest"] is None:
            return "no dictionary entry close enough for difflib to suggest one"
        return (f"closest dictionary entry is {finding['closest']!r} "
                f"(score {finding['score']:.2f}), below the "
                f"{FUZZY_MATCH_CUTOFF:.2f} cutoff the real pipeline fixes above")
    return "not a number, and clean_one_value() could not make sense of it alone"


def chapter_label_findings(chapter):
    """LABEL_HITS for one chapter, deduplicated within that chapter only -
    unlike aggregate_labels(), which merges the same label across chapters."""
    seen = {}
    for hit in LABEL_HITS:
        if hit["chapter"] != chapter:
            continue
        key = (hit["scope"], hit["col_ar"], hit["raw"])
        seen.setdefault(key, hit)
    return list(seen.values())


def chapter_value_findings(chapter):
    """VALUE_HITS for one chapter, deduplicated by raw text, keeping the first
    occurrence's row (country/indicator/year) as the example."""
    seen = {}
    order = []
    for hit in VALUE_HITS:
        if hit["chapter"] != chapter:
            continue
        if hit["raw"] not in seen:
            seen[hit["raw"]] = {**hit, "locations": []}
            order.append(hit["raw"])
        seen[hit["raw"]]["locations"].append((hit["file"], hit["sheet"]))
    return [seen[key] for key in order]


def write_chapter_reports(chapters, folder=None):
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    written = []

    for chapter in chapters:
        labels = chapter_label_findings(chapter)
        values = chapter_value_findings(chapter)
        structure = [row for row in STRUCTURE_PROBLEMS if row["chapter"] == chapter]

        lines = [
            f"DATA QUALITY ISSUES BEFORE PIPELINE EXECUTION - {chapter}",
            "=" * 78,
            f"generated {stamp}, from the raw questionnaires. Read-only summary -",
            "every outstanding issue lives here; correct it directly (the source",
            "questionnaire, translation dict_V2.xlsx, or value corrections.xlsx) and",
            "tell Claude to resume. Nothing here is ever applied automatically.",
            "",
            f"{len(labels)} label issue(s), {len(values)} value issue(s), "
            f"{len(structure)} structural issue(s)",
        ]

        if not (labels or values or structure):
            lines += ["", "Nothing found."]

        if labels:
            lines += ["", "-" * 78, "LABELS NOT IN THE DICTIONARY, EXACTLY", "-" * 78]
            for i, hit in enumerate(labels, start=1):
                lines.append(
                    f"{i}. Country: -  |  Sheet: {hit['file']} ({hit['sheet']})  |  "
                    f"Indicator: -  |  Year: -"
                )
                lines.append(f"   {hit['raw']!r} - {issue_reason(hit)}")

        if values:
            lines += ["", "-" * 78, "VALUES clean_one_value() COULD NOT PARSE", "-" * 78]
            for i, finding in enumerate(values, start=1):
                where = format_locations(finding["locations"])
                lines.append(
                    f"{i}. Country: {finding.get('country', '-')}  |  Sheet: {where}  |  "
                    f"Indicator: {finding.get('indicator', '-')}  |  "
                    f"Year: {finding.get('year', '-')}"
                )
                lines.append(f"   {finding['raw']!r} - {issue_reason(finding)}")

        if structure:
            lines += ["", "-" * 78, "STRUCTURAL PROBLEMS (fix in the source Excel file)", "-" * 78]
            for i, row in enumerate(structure, start=1):
                lines.append(
                    f"{i}. Country: -  |  Sheet: {row['file']} ({row['sheet']})  |  "
                    f"Indicator: -  |  Year: -"
                )
                lines.append(f"   {issue_reason(row)}")

        lines.append("")
        path = (folder or chapter_logs_folder(chapter)) / f"{CHAPTER_REPORT_PREFIX}{chapter}.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n".join(lines), encoding="utf-8")
        written.append(path)

    return written


## Run - Part 1

In [ ]:
"""
CELL: Main run - read every questionnaire and external file, write each
chapter's own issue report.
"""
run_checks()

chapter_report_paths = write_chapter_reports(chapters_to_process())
print("\n" + "=" * 70)
print("PER-CHAPTER ISSUE REPORTS WRITTEN")
print("=" * 70)
for p in chapter_report_paths:
    print(f"  {p}")

print("\n" + "#" * 70)
print("# This is the pipeline's first step. Review each file above and")
print("# correct directly whatever it calls for - the source questionnaire,")
print("# translation dict_V2.xlsx, source translations_V2.xlsx, or value")
print("# corrections.xlsx - before notebook 1 runs. Anything left as it is")
print("# reaches notebook 1 unchanged, and is reported again there.")
print("#" * 70)
